# Lunar Lander Agent with Stable-Baselines3

Train Deep Reinforcement Agent in Gymnasium environment with Stable-Baselines3 Proximal Policy Optimizationreinformcent library to be able to land the Lunar Lander to the Landing Pad by controlling left, right and main orientation engine

## Instal Dependencies

In [ ]:
!apt-get update
!apt-get install swig # Simplified Wrapper and Interface Generator is dependency for box2d-py for Gymnasium Box2D
!apt-get install -y build-essential python3-dev

In [ ]:
# libraries for creation of virtual screen
!apt install python3-opengl
!apt install ffmpeg
!apt install xvfb
!apt install cmake

In [ ]:
!pip install pyglet==1.5
!pip install pyvirtualdisplay

In [ ]:
!pip install imageio-ffmpeg
!pip install huggingface_hub

In [ ]:
!pip install 'stable-baselines3[extra]'

In [ ]:
!pip install huggingface-sb3

In [ ]:
!pip install "gymnasium[box2d]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/DeepRL/intro_deep_q_learning

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/DeepRL/intro_deep_q_learning


## Imports

In [ ]:
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Environment

### How it works

In [ ]:
# create an environment
env = gym.make("LunarLander-v3")

# reset this environment
observation, info = env.reset()

# example of taking 20 actions
for _ in range(20):
  # take a random action
  action = env.action_space.sample()
  print("Action taken:", action)

  # after executing the action, get next_state, reward, terminated, truncated and info
  observation, reward, terminated, truncated, info = env.step(action)

  # If the game is terminated (in our case we land, crashed) or truncated (timeout)
  if terminated or truncated:
      # reset the environment to its inital state whe episode is finished or terminated
      print("Environment is reset")
      observation, info = env.reset()

env.close()

Action taken: 2
Action taken: 1
Action taken: 2
Action taken: 3
Action taken: 3
Action taken: 0
Action taken: 2
Action taken: 3
Action taken: 3
Action taken: 2
Action taken: 1
Action taken: 2
Action taken: 3
Action taken: 2
Action taken: 3
Action taken: 1
Action taken: 0
Action taken: 2
Action taken: 1
Action taken: 3


/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

### Lunar Lander Env

**Observation Space** is a vector of size 8:

* Horizontal pad coordinate (x)
* Vertical pad coordinate (y)
* Horizontal speed (x)
* Vertical speed (y)
* Angle
* Angular speed
* If the left leg contact point has touched the land boolean
* If the right leg contact point has touched the land boolean

**Action Space** is 4 possible actions:

* Action 0: Do nothing,
* Action 1: Fire left orientation engine,
* Action 2: Fire the main engine,
* Action 3: Fire right orientation engine.

After every step a reward is granted. The total reward of an episode is the sum of the rewards for all the steps within that episode.

**Reward** for each step:

Is increased/decreased the closer/further the lander is to the landing pad.
Is increased/decreased the slower/faster the lander is moving.
Is decreased the more the lander is tilted (angle not horizontal).
Is increased by 10 points for each leg that is in contact with the ground.
Is decreased by 0.03 points each frame a side engine is firing.
Is decreased by 0.3 points each frame the main engine is firing.
The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.

In [ ]:
env = gym.make("LunarLander-v3")
env.reset()

print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample())

print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample())

_____OBSERVATION SPACE_____ 

Observation Space Shape (8,)
Sample observation [ 1.0308173  -2.0926704  -5.9679527   2.7732108  -4.098473   -1.8520626
  0.81747746  0.4219574 ]

 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 2


Vectorized Environment by stacking 16 independent environments into a single environment

In [ ]:
env = make_vec_env('LunarLander-v3', n_envs=16)

## Model

In [ ]:
# instantiate the agent with MultiLayerPerceptron since the input is a vector, use CnnPolicy with frames as input
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)


Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Train

In [ ]:
# train for 1,000,000 timesteps
model.learn(total_timesteps=1_000_000)

# save the model, LunarLander-v2 environment is depricated but Hugging Face require
# model and named environment LunarLander-v2
model_name = "ppo-LunarLander-v2"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 93.6     |
|    ep_rew_mean     | -198     |
| time/              |          |
|    fps             | 2976     |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 16384    |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 91.9         |
|    ep_rew_mean          | -142         |
| time/                   |              |
|    fps                  | 2294         |
|    iterations           | 2            |
|    time_elapsed         | 14           |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0063630557 |
|    clip_fraction        | 0.0487       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | 0.000676     |
|    learning_r

In [ ]:
# continue trainig
model.learn(total_timesteps=1_000_000)

# save the model
model_name = "ppo-LunarLander-v2"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 289      |
|    ep_rew_mean     | 277      |
| time/              |          |
|    fps             | 2102     |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 291         |
|    ep_rew_mean          | 273         |
| time/                   |             |
|    fps                  | 1602        |
|    iterations           | 2           |
|    time_elapsed         | 20          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.004243332 |
|    clip_fraction        | 0.0378      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.823      |
|    explained_variance   | 0.943       |
|    learning_rate        | 0.

## Evaluate

In [ ]:
# create an evaluation environment
eval_env = Monitor(gym.make("LunarLander-v3"))

# evaluate the model with 25 episodes and determinitic=True
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=25, deterministic=True)

print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=279.48 +/- 20.826807367290254


## Publish

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env

from huggingface_sb3 import package_to_hub

In [ ]:
repo_id = "elliemci/ppo-LunarLander-v2"

env_id = "LunarLander-v3"
# create an evaluation env and set the render_mode="rgb_array"
eval_env = DummyVecEnv([lambda: Monitor(gym.make(env_id, render_mode="rgb_array"))])
# model PPO MlpPolicy architecture
model_architecture = "PPO"
commit_message = "SB3 trained LunarLander-v2"

In [ ]:
package_to_hub(model=model,
               model_name=model_name,
               model_architecture=model_architecture,
               env_id=env_id,
               eval_env=eval_env,
               repo_id=repo_id,
               commit_message=commit_message)

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.
Saving video to /tmp/tmpnvl_1z36/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmpnvl_1z36/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpnvl_1z36/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmpnvl_1z36/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo elliemci/ppo-LunarLander-v2 to the Hugging Face Hub


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-v2/pytorch_variables.pth: 100%|##########| 1.26kB / 1.26kB            

  ...r-v2/policy.optimizer.pth: 100%|##########| 88.7kB / 88.7kB            

  ...LunarLander-v2/policy.pth: 100%|##########| 44.1kB / 44.1kB            

  ...ud/ppo-LunarLander-v2.zip: 100%|##########|  150kB /  150kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/elliemci/ppo-LunarLander-v2/tree/main/


CommitInfo(commit_url='https://huggingface.co/elliemci/ppo-LunarLander-v2/commit/0fdfe8fecfd2a7ca03ad60d4b19dffa2f9ebf842', commit_message='SB3 trained LunarLander-v2', commit_description='', oid='0fdfe8fecfd2a7ca03ad60d4b19dffa2f9ebf842', pr_url=None, repo_url=RepoUrl('https://huggingface.co/elliemci/ppo-LunarLander-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='elliemci/ppo-LunarLander-v2'), pr_revision=None, pr_num=None)

## Video

In [ ]:
import os
import base64
import torch
import numpy as np

from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv

from pathlib import Path
from IPython import display as ipythondisplay

os.system("Xvfb :1 -screen 0 1024x768x24 &")
os.environ['DISPLAY'] = ':1'


In [ ]:
def record_video(env_id, model, video_length=500, prefix="", video_folder="videos/"):
    """
    :param env_id: (str)
    :param model: (RL model) - custom Agent
    :param video_length: (int)
    :param prefix: (str)
    :param video_folder: (str)
    """
    # create a single environment for evaluation and rendering
    eval_env = gym.make(env_id, render_mode="rgb_array")
    # wrap the environment with DummyVecEnv as VecVideoRecorder expects a VecEnv
    eval_env = DummyVecEnv([lambda: eval_env])

    # start the video at step=0 and record video_length steps
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda step: step == 0,
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = eval_env.reset() # reset the environment and get the initial observation (unpacking one value)

    for _ in range(video_length):
        # get action from agent model
        # Use model.predict to get the action
        action, _states = model.predict(obs, deterministic=True)


        step_output = eval_env.step(action) # capture the output of step

        # adjust based on the print output
        obs, reward, terminated, info = step_output # Attempt to unpack 5 values

        done = terminated #or truncated # Combine terminated and truncated

        # If the episode is done, reset the environment for the next recording if video_length is long enough
        if done:
             obs = eval_env.reset() # Reset the environment for the next episode if done (unpacking one value)


    # close the video recorder
    eval_env.close()

In [ ]:
def show_videos(video_path="", prefix=""):
    """
    :param video_path: (str) Path to the folder containing videos
    :param prefix: (str) Filter the video, showing only the only starting with this prefix
    """
    html = []
    for mp4 in Path(video_path).glob("{}*.mp4".format(prefix)):
        video_b64 = base64.b64encode(mp4.read_bytes())
        html.append(
            """<video alt="{}" autoplay
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{}" type="video/mp4" />
                </video>""".format(
                mp4, video_b64.decode("ascii")
            )
        )
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))



In [ ]:
record_video("LunarLander-v3", model, video_length=500, prefix="ppo-lunarlander")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saving video to /content/drive/MyDrive/ColabNotebooks/DeepRL/intro_deep_q_learning/videos/ppo-lunarlander-step-0-to-step-500.mp4


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Moviepy - Building video /content/drive/MyDrive/ColabNotebooks/DeepRL/intro_deep_q_learning/videos/ppo-lunarlander-step-0-to-step-500.mp4.
Moviepy - Writing video /content/drive/MyDrive/ColabNotebooks/DeepRL/intro_deep_q_learning/videos/ppo-lunarlander-step-0-to-step-500.mp4



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/ColabNotebooks/DeepRL/intro_deep_q_learning/videos/ppo-lunarlander-step-0-to-step-500.mp4


In [ ]:
show_videos("videos", prefix="ppo")

## Manually uploade video

In [ ]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="./videos/ppo-lunarlander-step-0-to-step-500.mp4",
    path_in_repo="replay.mp4",
    repo_id="elliemci/ppo-LunarLander-v2",
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/elliemci/ppo-LunarLander-v2/commit/6fad6aa089717fe1daadbc64f8c94eb3ba2f6e11', commit_message='Upload replay.mp4 with huggingface_hub', commit_description='', oid='6fad6aa089717fe1daadbc64f8c94eb3ba2f6e11', pr_url=None, repo_url=RepoUrl('https://huggingface.co/elliemci/ppo-LunarLander-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='elliemci/ppo-LunarLander-v2'), pr_revision=None, pr_num=None)